# Experiment 10: Turbulent Pipe-Flow Model Comparison

**Aim:** Compare radial eddy-viscosity closures through velocity profiles, friction factors, pressure drop, and turbulent viscosity.

**Numerical method:** Axisymmetric radial momentum solution with laminar, mixing-length, Van-Driest, equilibrium k-epsilon, and equilibrium k-omega/SST surrogate closures.

> **Scope:** These are equilibrium radial closures, not complete two-equation RANS transport simulations.

This Google Colab notebook is self-contained. It reconstructs the tested solver inside the temporary Colab runtime, runs it, displays quantitative plots and an inline field contour, and packages every output file for download.

> **Colab workflow:** Choose a run mode, use **Runtime > Run all**, then download the results ZIP from the final cell. `quick` is only a workflow check; use `lab` or an approved `custom` case for submission.

In [ ]:
#@title 1. Prepare the self-contained Colab runtime

from pathlib import Path
import shutil
import subprocess
import sys
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None


def run_solver(script_path, arguments, output_dir, timeout=1800):
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(script_path), *[str(value) for value in arguments]]
    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=output_dir, capture_output=True, text=True, timeout=timeout)
    (output_dir / "solver_stdout.log").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "solver_stderr.log").write_text(completed.stderr, encoding="utf-8")
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or f"Solver exited with code {completed.returncode}")
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return output_dir


def read_csv(path):
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    return np.atleast_1d(data)


def show_summary(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    display(Markdown(f"### Solver summary\n```text\n{text}\n```"))
    return text


def list_outputs(output_dir):
    rows = []
    for path in sorted(Path(output_dir).iterdir()):
        if path.is_file():
            size = path.stat().st_size
            if size < 1024:
                size_text = f"{size} B"
            elif size < 1024**2:
                size_text = f"{size / 1024:.1f} KB"
            else:
                size_text = f"{size / 1024**2:.1f} MB"
            rows.append((path.name, size_text))
    table = "\n".join(f"| `{name}` | {size} |" for name, size in rows)
    display(Markdown(f"### Generated files\n| File | Size |\n|---|---:|\n{table}"))
    return rows


def read_structured_vtk(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    dims_index = next(i for i, line in enumerate(lines) if line.startswith("DIMENSIONS "))
    _, nx, ny, nz = lines[dims_index].split()
    nx, ny, nz = int(nx), int(ny), int(nz)
    if nz != 1:
        raise ValueError("Only 2D structured VTK files are supported by this notebook preview")
    points_index = next(i for i, line in enumerate(lines) if line.startswith("POINTS "))
    npoints = int(lines[points_index].split()[1])
    points = np.array([[float(v) for v in lines[points_index + 1 + k].split()[:3]] for k in range(npoints)])
    x = points[:, 0].reshape(ny, nx)
    y = points[:, 1].reshape(ny, nx)
    data_index = next(i for i, line in enumerate(lines) if line.startswith("POINT_DATA ")) + 1
    scalars = {}
    vectors = {}
    i = data_index
    while i < len(lines):
        if lines[i].startswith("SCALARS "):
            name = lines[i].split()[1]
            i += 2
            values = np.array([float(lines[i + k].split()[0]) for k in range(npoints)])
            scalars[name] = values.reshape(ny, nx)
            i += npoints
        elif lines[i].startswith("VECTORS "):
            name = lines[i].split()[1]
            i += 1
            values = np.array([[float(v) for v in lines[i + k].split()[:3]] for k in range(npoints)])
            vectors[name] = (values[:, 0].reshape(ny, nx), values[:, 1].reshape(ny, nx))
            i += npoints
        else:
            i += 1
    return x, y, scalars, vectors


def plot_vtk_scalar(path, preferred_field=None):
    path = Path(path)
    if not path.is_file():
        print(f"VTK preview skipped: {path.name} was not generated")
        return
    x, y, scalars, _ = read_structured_vtk(path)
    if not scalars:
        print("VTK preview skipped: no scalar field found")
        return
    field = preferred_field if preferred_field in scalars else next(iter(scalars))
    plt.figure(figsize=(8, 4.8))
    image = plt.contourf(x, y, scalars[field], levels=30)
    plt.colorbar(image, label=field.replace("_", " "))
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{field.replace('_', ' ').title()} - inline field preview")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()


def make_results_zip(output_dir, experiment_number):
    output_dir = Path(output_dir)
    archive = output_dir.parent / f"experiment_{experiment_number:02d}_results.zip"
    if archive.exists():
        archive.unlink()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file():
                handle.write(path, path.relative_to(output_dir))
    print(f"Results archive: {archive} ({archive.stat().st_size / 1024:.1f} KB)")
    return archive


EXPERIMENT = 10
ROOT = Path("/content/cfd_lab_exp10") if IN_COLAB else Path.cwd() / "_colab_runtime_exp10"
CODE_DIR = ROOT / "code"
OUTPUT_DIR = ROOT / "outputs"
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "common.py").write_text('from __future__ import annotations\n\nimport csv\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport numpy as np\n\n\ndef trapezoidal_integral(values: np.ndarray, coordinates: np.ndarray) -> float:\n    y = np.asarray(values, dtype=float).ravel()\n    x = np.asarray(coordinates, dtype=float).ravel()\n    if y.size != x.size or y.size < 2:\n        raise ValueError("trapezoidal integration requires equally sized arrays with at least two points")\n    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))\n\n\ndef write_csv(path: str | Path, header: Iterable[str], rows: Iterable[Iterable[object]]) -> None:\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(list(header))\n        writer.writerows(rows)\n\n\ndef mesh_xy(nx: int, ny: int, lx: float = 1.0, ly: float = 1.0, x0: float = 0.0, y0: float = 0.0) -> tuple[np.ndarray, np.ndarray]:\n    x = np.linspace(x0, x0 + lx, nx)\n    y = np.linspace(y0, y0 + ly, ny)\n    return np.meshgrid(x, y)\n\n\ndef write_structured_grid_vtk(\n    path: str | Path,\n    x: np.ndarray,\n    y: np.ndarray,\n    scalars: Mapping[str, np.ndarray] | None = None,\n    vectors: Mapping[str, tuple[np.ndarray, np.ndarray]] | None = None,\n    title: str = "CFD lab output",\n) -> None:\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y, dtype=float)\n    if x.shape != y.shape or x.ndim != 2:\n        raise ValueError("x and y must be equally shaped 2D arrays")\n    ny, nx = x.shape\n    scalar_map = dict(scalars or {})\n    vector_map = dict(vectors or {})\n    for name, values in scalar_map.items():\n        if np.asarray(values).shape != x.shape:\n            raise ValueError(f"scalar {name} has shape {np.asarray(values).shape}, expected {x.shape}")\n    for name, values in vector_map.items():\n        if len(values) != 2 or np.asarray(values[0]).shape != x.shape or np.asarray(values[1]).shape != x.shape:\n            raise ValueError(f"vector {name} has incompatible shape")\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET STRUCTURED_GRID\\n")\n        handle.write(f"DIMENSIONS {nx} {ny} 1\\n")\n        handle.write(f"POINTS {nx * ny} float\\n")\n        for j in range(ny):\n            for i in range(nx):\n                handle.write(f"{x[j, i]:.12e} {y[j, i]:.12e} 0\\n")\n        handle.write(f"POINT_DATA {nx * ny}\\n")\n        for name, values in scalar_map.items():\n            data = np.asarray(values, dtype=float)\n            handle.write(f"SCALARS {name} float 1\\n")\n            handle.write("LOOKUP_TABLE default\\n")\n            for value in data.ravel(order="C"):\n                handle.write(f"{value:.12e}\\n")\n        for name, values in vector_map.items():\n            vx = np.asarray(values[0], dtype=float)\n            vy = np.asarray(values[1], dtype=float)\n            handle.write(f"VECTORS {name} float\\n")\n            for ax, ay in zip(vx.ravel(order="C"), vy.ravel(order="C")):\n                handle.write(f"{ax:.12e} {ay:.12e} 0\\n")\n\n\ndef write_polyline_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, title: str = "polyline", close: bool = False) -> None:\n    x = np.asarray(x, dtype=float).ravel()\n    y = np.asarray(y, dtype=float).ravel()\n    if x.size != y.size or x.size < 2:\n        raise ValueError("polyline requires equally sized x and y arrays")\n    ids = list(range(x.size))\n    if close:\n        ids.append(0)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {x.size} float\\n")\n        for xi, yi in zip(x, y):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES 1 {len(ids) + 1}\\n")\n        handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_multiline_vtk(path: str | Path, lines: list[tuple[np.ndarray, np.ndarray]], title: str = "lines") -> None:\n    points: list[tuple[float, float]] = []\n    connectivity: list[list[int]] = []\n    for x, y in lines:\n        xa = np.asarray(x, dtype=float).ravel()\n        ya = np.asarray(y, dtype=float).ravel()\n        if xa.size != ya.size or xa.size < 2:\n            raise ValueError("each line needs equally sized x and y arrays")\n        ids = list(range(len(points), len(points) + xa.size))\n        points.extend(zip(xa.tolist(), ya.tolist()))\n        connectivity.append(ids)\n    size = sum(len(ids) + 1 for ids in connectivity)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {len(points)} float\\n")\n        for xi, yi in points:\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES {len(connectivity)} {size}\\n")\n        for ids in connectivity:\n            handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_points_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, scalars: Mapping[str, np.ndarray] | None = None, title: str = "points") -> None:\n    xa = np.asarray(x, dtype=float).ravel()\n    ya = np.asarray(y, dtype=float).ravel()\n    if xa.size != ya.size:\n        raise ValueError("point arrays must have equal size")\n    scalar_map = dict(scalars or {})\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {xa.size} float\\n")\n        for xi, yi in zip(xa, ya):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"VERTICES {xa.size} {2 * xa.size}\\n")\n        for i in range(xa.size):\n            handle.write(f"1 {i}\\n")\n        if scalar_map:\n            handle.write(f"POINT_DATA {xa.size}\\n")\n            for name, values in scalar_map.items():\n                data = np.asarray(values, dtype=float).ravel()\n                if data.size != xa.size:\n                    raise ValueError(f"scalar {name} has wrong size")\n                handle.write(f"SCALARS {name} float 1\\n")\n                handle.write("LOOKUP_TABLE default\\n")\n                for value in data:\n                    handle.write(f"{value:.12e}\\n")\n\n\ndef gradient(field: np.ndarray, dx: float, dy: float) -> tuple[np.ndarray, np.ndarray]:\n    gy, gx = np.gradient(field, dy, dx, edge_order=2)\n    return gx, gy\n\n\ndef laplacian(field: np.ndarray, dx: float, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    out[1:-1, 1:-1] = (\n        (field[1:-1, 2:] - 2.0 * field[1:-1, 1:-1] + field[1:-1, :-2]) / (dx * dx)\n        + (field[2:, 1:-1] - 2.0 * field[1:-1, 1:-1] + field[:-2, 1:-1]) / (dy * dy)\n    )\n    return out\n\n\ndef upwind_derivative_x(field: np.ndarray, velocity: np.ndarray, dx: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[1:-1, :-2]) / dx\n    forward = (field[1:-1, 2:] - field[1:-1, 1:-1]) / dx\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef upwind_derivative_y(field: np.ndarray, velocity: np.ndarray, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[:-2, 1:-1]) / dy\n    forward = (field[2:, 1:-1] - field[1:-1, 1:-1]) / dy\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef poisson_jacobi(phi: np.ndarray, rhs: np.ndarray, dx: float, dy: float, iterations: int, boundary: str = "neumann", mask: np.ndarray | None = None) -> np.ndarray:\n    dx2 = dx * dx\n    dy2 = dy * dy\n    denom = 2.0 * (dx2 + dy2)\n    for _ in range(iterations):\n        old = phi.copy()\n        phi[1:-1, 1:-1] = (\n            (old[1:-1, 2:] + old[1:-1, :-2]) * dy2\n            + (old[2:, 1:-1] + old[:-2, 1:-1]) * dx2\n            - rhs[1:-1, 1:-1] * dx2 * dy2\n        ) / denom\n        if mask is not None:\n            phi[mask > 0.5] = 0.0\n        if boundary == "neumann":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = 0.0\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n        elif boundary == "closed":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = phi[:, -2]\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n            phi -= np.mean(phi)\n        else:\n            raise ValueError("unsupported pressure boundary")\n    return phi\n\n\ndef tdma(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:\n    a = np.asarray(a, dtype=float).copy()\n    b = np.asarray(b, dtype=float).copy()\n    c = np.asarray(c, dtype=float).copy()\n    d = np.asarray(d, dtype=float).copy()\n    n = d.size\n    if n == 0 or a.size != n or b.size != n or c.size != n:\n        raise ValueError("TDMA arrays must have equal nonzero size")\n    for i in range(1, n):\n        if abs(b[i - 1]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        factor = a[i] / b[i - 1]\n        b[i] -= factor * c[i - 1]\n        d[i] -= factor * d[i - 1]\n    x = np.empty(n, dtype=float)\n    if abs(b[-1]) < 1e-30:\n        raise FloatingPointError("zero TDMA pivot")\n    x[-1] = d[-1] / b[-1]\n    for i in range(n - 2, -1, -1):\n        if abs(b[i]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        x[i] = (d[i] - c[i] * x[i + 1]) / b[i]\n    return x\n\n\ndef relative_l2(numerical: np.ndarray, reference: np.ndarray) -> float:\n    a = np.asarray(numerical, dtype=float)\n    b = np.asarray(reference, dtype=float)\n    denominator = float(np.linalg.norm(b.ravel()))\n    numerator = float(np.linalg.norm((a - b).ravel()))\n    return numerator / max(denominator, 1e-30)\n\n\ndef require_finite(*arrays: np.ndarray) -> None:\n    for array in arrays:\n        if not np.all(np.isfinite(array)):\n            raise FloatingPointError("non-finite numerical field detected")\n\n\ndef positive_float(value: str, name: str) -> float:\n    result = float(value)\n    if not math.isfinite(result) or result <= 0.0:\n        raise ValueError(f"{name} must be positive and finite")\n    return result\n\n\ndef positive_int(value: str, name: str, minimum: int = 1) -> int:\n    result = int(value)\n    if result < minimum:\n        raise ValueError(f"{name} must be at least {minimum}")\n    return result\n', encoding="utf-8")
(CODE_DIR / 'exp10_pipe_turbulence_models.py').write_text('from __future__ import annotations\n\nimport csv\nimport math\nimport sys\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom common import tdma, write_multiline_vtk, write_structured_grid_vtk\n\n\n@dataclass\nclass ModelResult:\n    name: str\n    reynolds: float\n    nu: float\n    friction: float\n    u_tau: float\n    pressure_drop: float\n    residual: float\n    iterations: int\n    radius: np.ndarray\n    velocity: np.ndarray\n    nut: np.ndarray\n    k: np.ndarray\n    epsilon: np.ndarray\n    omega: np.ndarray\n\n\ndef bulk_velocity(radius: np.ndarray, velocity: np.ndarray, pipe_radius: float, dr: float) -> float:\n    return 2.0 * float(np.sum(velocity * radius) * dr) / (pipe_radius * pipe_radius)\n\n\ndef solve_model(model: int, name: str, n: int, reynolds: float, max_iterations: int, tolerance: float) -> ModelResult:\n    pipe_radius = 0.5\n    diameter = 1.0\n    target_bulk = 1.0\n    nu = target_bulk * diameter / reynolds\n    dr = pipe_radius / n\n    radius = (np.arange(n) + 0.5) * dr\n    velocity = 2.0 * (1.0 - (radius / pipe_radius) ** 2)\n    nut = np.zeros(n)\n    k = np.zeros(n)\n    epsilon = np.zeros(n)\n    omega_field = np.zeros(n)\n    kappa = 0.41\n    a_plus = 26.0\n    c_mu = 0.09\n    beta_star = 0.09\n    a1 = 0.31\n    friction_initial = 0.3164 * max(reynolds, 4000.0) ** (-0.25)\n    u_tau = math.sqrt(max(friction_initial / 8.0, 1e-12))\n    residual = 1.0\n    completed = 0\n    for iteration in range(1, max_iterations + 1):\n        old = velocity.copy()\n        derivative = np.gradient(velocity, dr, edge_order=2)\n        wall_distance = np.maximum(pipe_radius - radius, 0.5 * dr)\n        y_plus = wall_distance * u_tau / nu\n        damping = 1.0 - np.exp(-y_plus / a_plus)\n        strain = np.abs(derivative)\n        k.fill(0.0)\n        epsilon.fill(0.0)\n        omega_field.fill(0.0)\n        if model == 0:\n            nut.fill(0.0)\n        elif model == 1:\n            mixing_length = kappa * wall_distance\n            nut = mixing_length * mixing_length * strain\n        elif model == 2:\n            mixing_length = kappa * wall_distance * damping\n            nut = mixing_length * mixing_length * strain\n        elif model == 3:\n            kdamp = 1.0 - np.exp(-y_plus / 12.0)\n            k = u_tau * u_tau / math.sqrt(c_mu) * kdamp * kdamp\n            y_use = np.maximum(wall_distance, 1.5 * nu / max(u_tau, 1e-12))\n            epsilon = c_mu ** 0.75 * np.maximum(k, 1e-16) ** 1.5 / (kappa * y_use)\n            nut = c_mu * k * k / np.maximum(epsilon, 1e-18) * damping * damping\n        elif model == 4:\n            kdamp = 1.0 - np.exp(-y_plus / 10.0)\n            k = u_tau * u_tau / math.sqrt(beta_star) * kdamp * kdamp\n            y_use = np.maximum(wall_distance, 1.5 * nu / max(u_tau, 1e-12))\n            omega_field = u_tau / (math.sqrt(beta_star) * kappa * y_use)\n            f2_arg = 2.0 * np.sqrt(np.maximum(k, 0.0)) / np.maximum(beta_star * omega_field * y_use, 1e-18)\n            f2 = np.tanh(np.maximum(f2_arg, 0.0) ** 2)\n            denominator = np.maximum(a1 * omega_field, strain * f2)\n            nut = a1 * k / np.maximum(denominator, 1e-18) * damping * damping\n            epsilon = beta_star * k * omega_field\n        else:\n            raise ValueError("unknown turbulence model")\n        nut = np.clip(nut, 0.0, 300.0 * nu)\n        effective = nu + nut\n        a = np.zeros(n)\n        b = np.zeros(n)\n        c = np.zeros(n)\n        rhs = radius * dr\n        for i in range(n):\n            diagonal = 0.0\n            if i > 0:\n                rw = i * dr\n                nuw = 0.5 * (effective[i] + effective[i - 1])\n                aw = rw * nuw / dr\n                a[i] = -aw\n                diagonal += aw\n            re = (i + 1) * dr\n            if i < n - 1:\n                nue = 0.5 * (effective[i] + effective[i + 1])\n                ae = re * nue / dr\n                c[i] = -ae\n                diagonal += ae\n            else:\n                ae = re * nu / (0.5 * dr)\n                diagonal += ae\n            b[i] = diagonal\n        candidate = tdma(a, b, c, rhs)\n        candidate *= target_bulk / max(bulk_velocity(radius, candidate, pipe_radius, dr), 1e-30)\n        wall_gradient = (0.0 - candidate[-1]) / (0.5 * dr)\n        new_u_tau = math.sqrt(max(nu * abs(wall_gradient), 1e-18))\n        relax = 1.0 if model == 0 else 0.45\n        velocity = relax * candidate + (1.0 - relax) * velocity\n        residual = float(np.linalg.norm(velocity - old) / max(np.linalg.norm(old), 1e-30))\n        u_tau = 0.35 * new_u_tau + 0.65 * u_tau\n        completed = iteration\n        if residual < tolerance and iteration > 20:\n            break\n    friction = 8.0 * u_tau * u_tau\n    pressure_drop = friction * 10.0 * 0.5\n    return ModelResult(name, reynolds, nu, friction, u_tau, pressure_drop, residual, completed, radius, velocity, nut, k, epsilon, omega_field)\n\n\ndef radial_interpolate(result: ModelResult, radial_position: np.ndarray) -> np.ndarray:\n    pipe_radius = 0.5\n    xp = np.concatenate(([0.0], result.radius, [pipe_radius]))\n    fp = np.concatenate(([result.velocity[0]], result.velocity, [0.0]))\n    return np.interp(radial_position, xp, fp)\n\n\ndef interpolate_field(radius: np.ndarray, values: np.ndarray, radial_position: np.ndarray, wall_value: float = 0.0) -> np.ndarray:\n    pipe_radius = 0.5\n    xp = np.concatenate(([0.0], radius, [pipe_radius]))\n    fp = np.concatenate(([values[0]], values, [wall_value]))\n    return np.interp(radial_position, xp, fp)\n\n\ndef main() -> int:\n    n_radial = int(sys.argv[1]) if len(sys.argv) > 1 else 121\n    vtk_nx = int(sys.argv[2]) if len(sys.argv) > 2 else 160\n    vtk_ny = int(sys.argv[3]) if len(sys.argv) > 3 else 61\n    re_min = float(sys.argv[4]) if len(sys.argv) > 4 else 10000.0\n    re_max = float(sys.argv[5]) if len(sys.argv) > 5 else 100000.0\n    n_re = int(sys.argv[6]) if len(sys.argv) > 6 else 3\n    max_iterations = int(sys.argv[7]) if len(sys.argv) > 7 else 1500\n    tolerance = float(sys.argv[8]) if len(sys.argv) > 8 else 1e-7\n    if n_radial < 21 or vtk_nx < 10 or vtk_ny < 11 or re_min <= 0.0 or re_max < re_min or n_re < 1 or max_iterations < 1 or tolerance <= 0.0:\n        raise ValueError("usage: python exp10_pipe_turbulence_models.py nRadial vtkNx vtkNy ReMin ReMax nRe maxIterations tolerance")\n    models = [(0, "laminar"), (1, "mixing_length"), (2, "van_driest"), (3, "kepsilon"), (4, "komega_sst")]\n    reynolds_values = np.array([re_min]) if n_re == 1 else np.geomspace(re_min, re_max, n_re)\n    summary_rows: list[list[object]] = []\n    for reynolds in reynolds_values:\n        results: list[ModelResult] = []\n        for model, name in models:\n            result = solve_model(model, name, n_radial, float(reynolds), max_iterations, tolerance)\n            results.append(result)\n            blasius = 0.3164 * reynolds ** (-0.25)\n            haaland = 1.0 / (-1.8 * math.log10(6.9 / reynolds)) ** 2\n            summary_rows.append([reynolds, name, result.friction, blasius, haaland, result.u_tau, result.pressure_drop, result.velocity[0], float(np.max(result.nut / result.nu)), result.iterations, result.residual])\n            print(f"Re={reynolds:.0f} model={name} friction={result.friction:.6e} U_center={result.velocity[0]:.6e} residual={result.residual:.6e}")\n        x1 = np.linspace(0.0, 10.0, vtk_nx)\n        y1 = np.linspace(-0.5, 0.5, vtk_ny)\n        x, y = np.meshgrid(x1, y1)\n        radial_position = np.abs(y)\n        scalars: dict[str, np.ndarray] = {}\n        vectors: dict[str, tuple[np.ndarray, np.ndarray]] = {}\n        for result in results:\n            velocity_profile = radial_interpolate(result, radial_position)\n            nut_ratio = interpolate_field(result.radius, result.nut / result.nu, radial_position)\n            k_field = interpolate_field(result.radius, result.k, radial_position)\n            epsilon_field = interpolate_field(result.radius, result.epsilon, radial_position)\n            omega_field = interpolate_field(result.radius, result.omega, radial_position)\n            pressure = result.pressure_drop * (1.0 - x / 10.0)\n            scalars[f"U_{result.name}"] = velocity_profile\n            scalars[f"nut_over_nu_{result.name}"] = nut_ratio\n            scalars[f"pressure_{result.name}"] = pressure\n            scalars[f"k_{result.name}"] = k_field\n            scalars[f"epsilon_{result.name}"] = epsilon_field\n            scalars[f"omega_{result.name}"] = omega_field\n            vectors[f"velocity_{result.name}"] = (velocity_profile, np.zeros_like(velocity_profile))\n        tag = f"{reynolds:.0f}"\n        write_structured_grid_vtk(f"pipe2d_compare_Re{tag}.vtk", x, y, scalars, vectors, "2D axisymmetric pipe turbulence-model comparison")\n        with open(f"pipe2d_profiles_Re{tag}.csv", "w", newline="", encoding="utf-8") as handle:\n            writer = csv.writer(handle)\n            header = ["r_over_R"]\n            for result in results:\n                header.extend([f"U_{result.name}", f"nut_over_nu_{result.name}", f"k_{result.name}", f"epsilon_{result.name}", f"omega_{result.name}"])\n            writer.writerow(header)\n            for i in range(n_radial):\n                row: list[float] = [results[0].radius[i] / 0.5]\n                for result in results:\n                    row.extend([result.velocity[i], result.nut[i] / result.nu, result.k[i], result.epsilon[i], result.omega[i]])\n                writer.writerow(row)\n    with open("pipe2d_rans_summary.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["Re", "model", "friction_factor_Darcy", "Blasius_reference", "Haaland_smooth_reference", "uTau", "pressure_drop_over_10D", "U_center", "maximum_nut_over_nu", "iterations", "residual"])\n        writer.writerows(summary_rows)\n    write_multiline_vtk("pipe2d_walls.vtk", [(np.array([0.0, 10.0]), np.array([-0.5, -0.5])), (np.array([0.0, 10.0]), np.array([0.5, 0.5]))], "Pipe wall lines")\n    with open("pipe2d_rans_summary.txt", "w", encoding="utf-8") as handle:\n        handle.write("Teaching-level 2D axisymmetric fully developed pipe-flow turbulence-model comparison\\n")\n        handle.write("Models laminar mixing_length van_driest equilibrium_kepsilon equilibrium_komega_sst\\n")\n        handle.write("The model solves radial eddy-viscosity closures and extrudes the profile along the pipe axis.\\n")\n        handle.write("This is not a full industrial two-equation RANS transport solver.\\n")\n        handle.write("Compare friction factors with Blasius and Haaland columns in pipe2d_rans_summary.csv.\\n")\n    print("Wrote pipe2d_compare_Re*.vtk, pipe2d_profiles_Re*.csv, pipe2d_rans_summary.csv, pipe2d_rans_summary.txt, pipe2d_walls.vtk")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
SCRIPT = CODE_DIR / 'exp10_pipe_turbulence_models.py'
print("Runtime:", "Google Colab" if IN_COLAB else "local notebook test")
print("Working folder:", ROOT)
print("No package installation or external file upload is required.")


## Parameters

Use `quick` to test the workflow, `lab` for the prescribed case, and `custom` only when changing the form values for the assignment.

In [ ]:
#@title 2. Select run mode and parameters
RUN_MODE = "quick" #@param ["quick", "lab", "custom"]

n_radial = 121 #@param {type:"integer"}
vtk_nx = 160 #@param {type:"integer"}
vtk_ny = 61 #@param {type:"integer"}
Re_min = 10000.0 #@param {type:"number"}
Re_max = 100000.0 #@param {type:"number"}
n_Re = 3 #@param {type:"integer"}
max_iterations = 1500 #@param {type:"integer"}
tolerance = 1e-07 #@param {type:"number"}

QUICK_ARGS = [21, 30, 21, 10000, 12000, 1, 8, 0.0001]
LAB_ARGS = [121, 160, 61, 10000, 100000, 3, 1500, 1e-07]
CUSTOM_ARGS = [n_radial, vtk_nx, vtk_ny, Re_min, Re_max, n_Re, max_iterations, tolerance]
ARGS = QUICK_ARGS if RUN_MODE == "quick" else LAB_ARGS if RUN_MODE == "lab" else CUSTOM_ARGS
print("Run mode:", RUN_MODE)
print("Arguments:", ARGS)


## Run the solver

In [ ]:
#@title 3. Run the numerical solver
OUT = run_solver(SCRIPT, ARGS, OUTPUT_DIR)


## Numerical summary

In [ ]:
#@title 4. Read the solver summary
summary_text = show_summary(OUT / 'pipe2d_rans_summary.txt')


## Quantitative plots

In [ ]:
#@title 5. Plot the primary numerical results
summary = read_csv(OUT / "pipe2d_rans_summary.csv")
plt.figure(figsize=(8, 4))
for model in np.unique(summary["model"]):
    mask = summary["model"] == model
    plt.loglog(summary["Re"][mask], summary["friction_factor_Darcy"][mask], marker="o", label=str(model))
reference_re = np.unique(summary["Re"])
reference_blasius = np.array([summary["Blasius_reference"][summary["Re"] == value][0] for value in reference_re])
reference_haaland = np.array([summary["Haaland_smooth_reference"][summary["Re"] == value][0] for value in reference_re])
plt.loglog(reference_re, reference_blasius, "k--", label="Blasius")
plt.loglog(reference_re, reference_haaland, "k-.", label="Haaland")
plt.xlabel("Reynolds number")
plt.ylabel("Darcy friction factor")
plt.title("Friction-factor comparison")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()

profile_files = sorted(OUT.glob("pipe2d_profiles_Re*.csv"))
profiles = read_csv(profile_files[0])
plt.figure(figsize=(8, 4))
for field in profiles.dtype.names:
    if field.startswith("U_"):
        plt.plot(profiles[field], profiles["r_over_R"], label=field[2:])
plt.xlabel("Mean velocity")
plt.ylabel("r/R")
plt.title(f"Velocity profiles: {profile_files[0].stem}")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()

## Inline field visualization

This preview removes the need for ParaView during the basic laboratory workflow. The VTK file is still included in the downloaded results for optional advanced post-processing.

In [ ]:
#@title 6. Display the final field and generated files
plot_vtk_scalar(OUT / 'pipe2d_compare_Re10000.vtk', 'U_mixing_length')
files = list_outputs(OUT)


## Download results

In [ ]:
#@title 7. Package and download the results
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
RESULTS_ZIP = make_results_zip(OUT, 10)
if DOWNLOAD_RESULTS:
    if IN_COLAB:
        colab_files.download(str(RESULTS_ZIP))
    else:
        print("Local test mode. Archive retained at:", RESULTS_ZIP)
else:
    print("Set DOWNLOAD_RESULTS to True and run this cell when ready to download.")


## 7. Compulsory assignment

Compare all model profiles and $\nu_t/\nu$ at $Re=10^4$, approximately $3.16\times10^4$, and $10^5$. Identify the fullest profile and compare friction factors with Blasius and Haaland.

## Report checklist

- State the governing equations and boundary conditions.
- Report the complete parameter set and grid.
- Include residual or error history.
- Include the required quantitative comparison.
- Include at least one correctly labelled inline field figure; VTK/ParaView figures are optional unless specifically assigned.
- Distinguish observed numerical behaviour from physical interpretation.
- State the limitations of the supplied teaching model.